In [0]:
# =====================================
# NOTEBOOK : NB_GOLD_FACT_PAYMENT
# LAYER    : GOLD
# TARGET   : FACT_PAYMENT
# =====================================

In [0]:
%run ../COMMON/NB_CONFIG

In [0]:
%run ../COMMON/NB_CONNECTION

In [0]:
%run ../COMMON/NB_UTILS

In [0]:
payment_df = spark.table(ods_payment_table).filter(F.col("IS_CURRENT") == "Y")

fact_payment_df = (
    payment_df
    .withColumn("DATE_ID", F.date_format(F.col("PAYMENT_DATE"), "yyyyMMdd"))
    .withColumn("CLAIM_ID", F.col("CLAIM_ID").cast("bigint"))   # ✅ align with FACT_CLAIM
    .select(
        "PAYMENT_ID",          # STRING
        "CLAIM_ID",            # BIGINT
        "DATE_ID",             # Surrogate key for DIM_DATE
        "PAYMENT_AMOUNT",      # DECIMAL(18,2)
        "PAYMENT_YEAR",        # INT
        "PAYMENT_MONTH",       # INT
        "PAYMENT_DELAY_DAYS",  # INT
        "STATUS_FLAG"          # STRING
    )
)

fact_payment_df.write.format("delta").mode("overwrite").saveAsTable(fact_payment_table)